# Build a model

Are the signals from `pbs-1`, `pbs-2`, and `pbs-3` sufficiently discrimatory of human breast cancer?

## 0. Initializations

In [1]:
## 0. Initializations
# -- imports --
import anndata as ad
import gseapy as gp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns

import random

from collections import defaultdict

from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from statistics import median

from signals_in_the_noise.analysis.noise_phenotypes import classify_noise_subtypes
from signals_in_the_noise.preprocessing.gse161529 import GSE161529

In [2]:
# -- datasets --
gse = GSE161529()

## 1. Preprocessing Data

1. Subset data to samples that have a potential biological signal.
2. Determine the distribution of number of samples per subject - use to downsample for parity.
3. Determine the distribution of number of cells per subject - use to downsample for parity

In [3]:
# -- subset data --
subtype_to_adata = defaultdict(dict)
choices = ["pbs-1", "pbs-2", "pbs-3"]
for adata in gse.objects.values():
    adata_noise = adata[adata.obs["is_noise"] == 1].copy()
    classify_noise_subtypes(adata_noise)
    conditions = [
        adata_noise.obs["pbs-1"] == 1,
        adata_noise.obs["pbs-2"] == 1,
        adata_noise.obs["pbs-3"] == 1,
    ]
    adata_noise.obs["pbs"] = np.select(conditions, choices, default="none")
    adata_pbs_only = adata_noise[adata_noise.obs["pbs"] != "none"]
    subtype_to_adata[adata_pbs_only.uns['cancer_type']].update({
        adata_pbs_only.uns['title'] : adata_pbs_only
    })

In [4]:
subtype_to_adata.keys()

dict_keys(['Normal', 'BRCA1 pre-neoplastic', 'Triple negative tumour', 'Triple negative BRCA1 tumour', 'HER2+ tumour', 'PR+ tumour', 'ER+ tumour'])

In [5]:
def print_data_stats(adata_collection):
    results = []
    for subtype, adatas in adata_collection.items():
        num_subjects = len(adatas)
        num_samples = [adata.n_obs for adata in adatas.values()]
        print(f"{subtype} - {num_samples}")
        results.append((subtype, num_subjects, min(num_samples), median(num_samples)))
    
    return pd.DataFrame(results, columns=['Cancer sub-type', '# Subjects', 'Min # Samples', 'Median # Samples'])

print_data_stats(subtype_to_adata)

Normal - [72, 136, 10, 272, 512, 28, 14, 46, 21, 86, 55, 52, 93, 23, 29, 116, 57, 149, 69, 88, 63, 80, 253, 178]
BRCA1 pre-neoplastic - [79, 113, 138, 146]
Triple negative tumour - [108, 96, 28, 69]
Triple negative BRCA1 tumour - [56, 107, 223, 668]
HER2+ tumour - [60, 619, 121, 12, 127, 1161]
PR+ tumour - [66]
ER+ tumour - [327, 271, 344, 134, 28, 105, 301, 79, 114, 142, 86, 83, 170, 30, 159, 81, 26, 3, 814, 610, 234, 99, 756, 31, 12]


,Cancer sub-type,# Subjects,Min # Samples,Median # Samples
0,Normal,24,10,70.5
1,BRCA1 pre-neoplastic,4,79,125.5
2,Triple negative tumour,4,28,82.5
3,Triple negative BRCA1 tumour,4,56,165.0
4,HER2+ tumour,6,12,124.0
5,PR+ tumour,1,66,66.0
6,ER+ tumour,25,3,114.0


### 1.a. Decisions

1. Exclude cancer sub-type `PR+ tumour` because there is only a single subject in the dataset.
2. Downsample subjects to `3` with a minimum of `50` samples.

### 1.b. Further subset data

Produces a dataset with the decisions.

In [6]:
def subset_data_for_modeling(require_min_subject=True, require_min_samples=True):
    results = {}
    for subtype, adatas in subtype_to_adata.items():
        if require_min_subject and subtype == 'PR+ tumour':
            # only has one subject, exclude
            continue
        valid_subjects_to_adata = defaultdict(dict)
        for adata in adatas.values():
            if require_min_samples and adata.n_obs < 50:
                # exclude subjects with fewer than 50 samples
                continue
            valid_subjects_to_adata.update({
                adata.uns['title']: adata,
            })
        results[subtype] = valid_subjects_to_adata
    return results
valid_adatas = subset_data_for_modeling(require_min_subject=False, require_min_samples=False)

In [7]:
print_data_stats(valid_adatas)

Normal - [72, 136, 272, 512, 86, 55, 52, 93, 116, 57, 149, 69, 88, 63, 80, 253, 178]
BRCA1 pre-neoplastic - [79, 113, 138, 146]
Triple negative tumour - [108, 96, 69]
Triple negative BRCA1 tumour - [56, 107, 223, 668]
HER2+ tumour - [60, 619, 121, 127, 1161]
ER+ tumour - [327, 271, 344, 134, 105, 301, 79, 114, 142, 86, 83, 170, 159, 81, 814, 610, 234, 99, 756]


,Cancer sub-type,# Subjects,Min # Samples,Median # Samples
0,Normal,17,52,88.0
1,BRCA1 pre-neoplastic,4,79,125.5
2,Triple negative tumour,3,69,96.0
3,Triple negative BRCA1 tumour,4,56,165.0
4,HER2+ tumour,5,60,127.0
5,ER+ tumour,19,79,159.0


### 1.3. Utility to Generate Dataset for Modeling

Creates a list of dataframe from the AnnDatas that represents all cancer sub-types (except for `PR+ tumour`), with the following characteristics:

1. `3` subjects per sub-type, `6` sub-types = `18` total subjects
2. `1` dataframe per subject
3. Columns
   1. `has_cancer`
   2. `cancer_type`
   3. `cell_population`
   4. `menopause_status`
   5. `pbs-1`
   6. `pbs-2`
   7. `pbs-3`

Separating the subjects to a row will simplify using the leave-one-out strategy.

In [8]:
def generate_dataframe_for_modeling(downsample=True):
    results = []
    for adatas in valid_adatas.values():
        if downsample:
            subjects = np.random.choice(list(adatas.keys()), 3)
        else:
            subjects = adatas.keys()

        for subject in subjects:
            adata = adatas[subject]
            df = pd.DataFrame({
                'pbs-1': adata.obs['pbs-1'].astype(int),
                'pbs-2': adata.obs['pbs-2'].astype(int),
                'pbs-3': adata.obs['pbs-3'].astype(int),
            })
            df['has_cancer'] = (adata.uns['cancer_type'] != 'Normal')
            df['cancer_type'] = str(adata.uns['cancer_type'])
            df['cell_population'] = str(adata.uns['cell_population'])
            df['menopause_status'] = str(adata.uns['menopause_status'])
            
            results.append(df)
    return results

In [9]:
generate_dataframe_for_modeling()[0]

,pbs-1,pbs-2,pbs-3,has_cancer,cancer_type,cell_population,menopause_status
AAAGTAGAGCTAGTCT-1,1,0,0,False,Normal,Total,Pre
AACTCTTTCTACTTAC-1,0,1,0,False,Normal,Total,Pre
AAGGAGCAGCTAACTC-1,0,0,1,False,Normal,Total,Pre
AAGGTTCTCAGCTCTC-1,0,1,0,False,Normal,Total,Pre
AATCCAGAGGTAGCCA-1,1,0,0,False,Normal,Total,Pre
...,...,...,...,...,...,...,...
TTAGTTCTCGGCTTGG-1,0,1,0,False,Normal,Total,Pre
TTGCGTCAGCCTATGT-1,1,0,0,False,Normal,Total,Pre
TTGGAACAGCGTAATA-1,0,1,0,False,Normal,Total,Pre
TTGGAACGTGTAATGA-1,1,0,0,False,Normal,Total,Pre


## 2. Build the classifier

In [10]:
def logistic_regression_loo_classifier(dfs, *, by_subjects=True, as_dict=False, model=None, target='cancer_type'):
    loo = LeaveOneOut()
    y_true, y_pred = [], []
    
    for train_idx, test_idx in loo.split(dfs):
        train_df = pd.concat([dfs[i] for i in train_idx])
        # there will only ever be one in the test
        test_df = dfs[test_idx[0]]
    
        X_train = train_df[['pbs-1', 'pbs-2', 'pbs-3']].values
        y_train = train_df[target].values
    
        X_test = test_df[['pbs-1', 'pbs-2', 'pbs-3']].values
        y_test = test_df[target].values

        if not model:
            model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)
    
        if by_subjects:
            counts = np.unique(model.predict(X_test), return_counts=True)
            # return the majority vote for the subject
            y_pred.append(counts[0][np.argmax(counts[1])])
            y_true.append(test_df[target].iloc[0])
        else:
            y_pred.extend(model.predict(X_test))
            y_true.extend(y_test)
    
    return classification_report(y_true, y_pred, output_dict=as_dict)

dfs = generate_dataframe_for_modeling()
print(logistic_regression_loo_classifier(dfs))

                              precision    recall  f1-score   support

        BRCA1 pre-neoplastic       0.00      0.00      0.00       3.0
                  ER+ tumour       0.00      0.00      0.00       3.0
                HER2+ tumour       0.00      0.00      0.00       3.0
                      Normal       0.00      0.00      0.00       3.0
Triple negative BRCA1 tumour       0.00      0.00      0.00       3.0
      Triple negative tumour       0.00      0.00      0.00       3.0

                    accuracy                           0.00      18.0
                   macro avg       0.00      0.00      0.00      18.0
                weighted avg       0.00      0.00      0.00      18.0



C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [11]:
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

reports = []
by_subject = False
for run in range(10):
    dfs = generate_dataframe_for_modeling()
    report = logistic_regression_loo_classifier(dfs, as_dict=True)
    reports.append(pd.DataFrame(report).T)

C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [12]:
avg_report = pd.concat(reports).groupby(level=0).mean()
std_report = pd.concat(reports).groupby(level=0).std()

print("Mean:")
print(avg_report)
print("\nStd:")
print(std_report)

Mean:
                              precision    recall  f1-score    support
BRCA1 pre-neoplastic           0.000000  0.000000  0.000000   3.000000
ER+ tumour                     0.029167  0.133333  0.046753   3.000000
HER2+ tumour                   0.081740  0.466667  0.139098   3.000000
Normal                         0.025000  0.033333  0.028571   3.000000
Triple negative BRCA1 tumour   0.006250  0.033333  0.010526   3.000000
Triple negative tumour         0.000000  0.000000  0.000000   3.000000
accuracy                       0.111111  0.111111  0.111111   0.111111
macro avg                      0.023693  0.111111  0.037491  18.000000
weighted avg                   0.023693  0.111111  0.037491  18.000000

Std:
                              precision    recall  f1-score   support
BRCA1 pre-neoplastic           0.000000  0.000000  0.000000  0.000000
ER+ tumour                     0.062268  0.322031  0.101561  0.000000
HER2+ tumour                   0.061530  0.358323  0.105003  0.00000

In [13]:
dfs = generate_dataframe_for_modeling(downsample=False)
print(logistic_regression_loo_classifier(dfs))

                              precision    recall  f1-score   support

        BRCA1 pre-neoplastic       0.00      0.00      0.00         4
                  ER+ tumour       0.37      1.00      0.54        19
                HER2+ tumour       0.00      0.00      0.00         5
                      Normal       0.00      0.00      0.00        17
Triple negative BRCA1 tumour       0.00      0.00      0.00         4
      Triple negative tumour       0.00      0.00      0.00         3

                    accuracy                           0.37        52
                   macro avg       0.06      0.17      0.09        52
                weighted avg       0.13      0.37      0.20        52



C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [14]:
dfs = generate_dataframe_for_modeling(downsample=False)
print(logistic_regression_loo_classifier(dfs, by_subjects=False))

                              precision    recall  f1-score   support

        BRCA1 pre-neoplastic       0.00      0.00      0.00       476
                  ER+ tumour       0.41      0.89      0.56      4909
                HER2+ tumour       0.00      0.00      0.00      2088
                      Normal       0.00      0.00      0.00      2331
Triple negative BRCA1 tumour       0.00      0.00      0.00      1054
      Triple negative tumour       0.00      0.00      0.00       273

                    accuracy                           0.39     11131
                   macro avg       0.07      0.15      0.09     11131
                weighted avg       0.18      0.39      0.25     11131



C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\silly\miniconda3\envs\signals-in-the-noise\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

### 2.1. Explore other models

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import warnings
from sklearn.exceptions import UndefinedMetricWarning

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB()
}

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    n_runs = 10
    for model_name, model in models.items():
        reports = []
        all_y_true, all_y_pred = [], []
        
        for run in range(n_runs):
            dfs = generate_dataframe_for_modeling(downsample=False)
            report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model)
            reports.append(pd.DataFrame(report).T)
    
        avg_report = pd.concat(reports).groupby(level=0).mean()
    
        print(f"{'='*25}")
        print(f"Model: {model_name}")
        print(f"{'-'*25}")
        print(f"Mean: {avg_report}")
        print()

Model: Logistic Regression
-------------------------
Mean:                               precision    recall  f1-score       support
BRCA1 pre-neoplastic           0.000000  0.000000  0.000000    476.000000
ER+ tumour                     0.412354  0.889387  0.563464   4909.000000
HER2+ tumour                   0.000000  0.000000  0.000000   2088.000000
Normal                         0.000000  0.000000  0.000000   2331.000000
Triple negative BRCA1 tumour   0.000000  0.000000  0.000000   1054.000000
Triple negative tumour         0.000000  0.000000  0.000000    273.000000
accuracy                       0.392238  0.392238  0.392238      0.392238
macro avg                      0.068726  0.148231  0.093911  11131.000000
weighted avg                   0.181856  0.392238  0.248499  11131.000000

Model: Decision Tree
-------------------------
Mean:                               precision    recall  f1-score       support
BRCA1 pre-neoplastic           0.000000  0.000000  0.000000    476.000000

### 2.2. Model ANY cancer

In [16]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    n_runs = 10
    for model_name, model in models.items():
        reports = []
        
        for run in range(n_runs):
            dfs = generate_dataframe_for_modeling(downsample=False)
            report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model, target="has_cancer")
            reports.append(pd.DataFrame(report).T)
    
        avg_report = pd.concat(reports).groupby(level=0).mean()
    
        print(f"{'='*25}")
        print(f"Model: {model_name}")
        print(f"{'-'*25}")
        print(f"Mean: {avg_report}")
        print()

Model: Logistic Regression
-------------------------
Mean:               precision    recall  f1-score       support
False          0.000000  0.000000  0.000000   2331.000000
True           0.790585  1.000000  0.883047   8800.000000
accuracy       0.790585  0.790585  0.790585      0.790585
macro avg      0.395292  0.500000  0.441523  11131.000000
weighted avg   0.625024  0.790585  0.698123  11131.000000

Model: Decision Tree
-------------------------
Mean:               precision    recall  f1-score       support
False          0.000000  0.000000  0.000000   2331.000000
True           0.790585  1.000000  0.883047   8800.000000
accuracy       0.790585  0.790585  0.790585      0.790585
macro avg      0.395292  0.500000  0.441523  11131.000000
weighted avg   0.625024  0.790585  0.698123  11131.000000

Model: KNN
-------------------------
Mean:               precision    recall  f1-score       support
False          0.209415  1.000000  0.346308   2331.000000
True           0.000000  0.0000

In [17]:
pd.concat(dfs).groupby('has_cancer')[['pbs-1','pbs-2','pbs-3']].mean()

,pbs-1,pbs-2,pbs-3
has_cancer,,,
False,0.286572,0.486486,0.226941
True,0.340568,0.265909,0.393523


In [18]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    n_runs = 10
    for model_name, model in models.items():
        reports = []
        
        for run in range(n_runs):
            dfs = generate_dataframe_for_modeling(downsample=False)
            report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model, target="has_cancer")
            reports.append(pd.DataFrame(report).T)
    
        avg_report = pd.concat(reports).groupby(level=0).mean()
    
        print(f"{'='*25}")
        print(f"Model: {model_name}")
        print(f"{'-'*25}")
        print(f"Mean: {avg_report}")
        print()

Model: Logistic Regression
-------------------------
Mean:               precision    recall  f1-score       support
False          0.000000  0.000000  0.000000   2331.000000
True           0.790585  1.000000  0.883047   8800.000000
accuracy       0.790585  0.790585  0.790585      0.790585
macro avg      0.395292  0.500000  0.441523  11131.000000
weighted avg   0.625024  0.790585  0.698123  11131.000000

Model: Decision Tree
-------------------------
Mean:               precision    recall  f1-score       support
False          0.000000  0.000000  0.000000   2331.000000
True           0.790585  1.000000  0.883047   8800.000000
accuracy       0.790585  0.790585  0.790585      0.790585
macro avg      0.395292  0.500000  0.441523  11131.000000
weighted avg   0.625024  0.790585  0.698123  11131.000000

Model: KNN
-------------------------
Mean:               precision    recall  f1-score       support
False          0.209415  1.000000  0.346308   2331.000000
True           0.000000  0.0000

#### 2.3.1 Need to downsample to proceed

There are much more cancer than normal, this is skewing the results.

There are only `17` normal subjects.

In [19]:
dfs = generate_dataframe_for_modeling(downsample=False)

In [20]:
dfs_cancer = []
dfs_normal = []

for df in dfs:
    if df['cancer_type'].unique()[0] == 'Normal':
        dfs_normal.append(df)
    else:
        dfs_cancer.append(df)

In [21]:
len(dfs_cancer), len(dfs_normal)

(35, 17)

In [22]:
downsample_indices = random.sample(range(len(dfs_cancer)), len(dfs_normal))

In [23]:
subset_cancer = [dfs_cancer[idx] for idx in downsample_indices]

In [24]:
dfs = dfs_normal.copy()
dfs.extend(subset_cancer)

balanced_df = pd.concat(dfs)
print(balanced_df['has_cancer'].value_counts())

has_cancer
True     3929
False    2331
Name: count, dtype: int64


In [25]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    n_runs = 10
    for model_name, model in models.items():
        reports = []
        for run in range(n_runs):
            report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model, target="has_cancer")
            reports.append(pd.DataFrame(report).T)
    
        avg_report = pd.concat(reports).groupby(level=0).mean()
    
        print(f"{'='*25}")
        print(f"Model: {model_name}")
        print(f"{'-'*25}")
        print(f"Mean: {avg_report}")
        print()

Model: Logistic Regression
-------------------------
Mean:               precision    recall  f1-score      support
False          0.160646  0.089661  0.115088  2331.000000
True           0.572091  0.722067  0.638389  3929.000000
accuracy       0.486581  0.486581  0.486581     0.486581
macro avg      0.366368  0.405864  0.376738  6260.000000
weighted avg   0.418884  0.486581  0.443530  6260.000000

Model: Decision Tree
-------------------------
Mean:               precision    recall  f1-score      support
False          0.160646  0.089661  0.115088  2331.000000
True           0.572091  0.722067  0.638389  3929.000000
accuracy       0.486581  0.486581  0.486581     0.486581
macro avg      0.366368  0.405864  0.376738  6260.000000
weighted avg   0.418884  0.486581  0.443530  6260.000000

Model: KNN
-------------------------
Mean:               precision    recall  f1-score      support
False          0.372364  1.000000  0.542661  2331.000000
True           0.000000  0.000000  0.000000  

In [26]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    n_runs = 1000
    for model_name, model in models.items():
        reports = []
        for run in range(n_runs):
            report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model, target="has_cancer")
            reports.append(pd.DataFrame(report).T)
    
        avg_report = pd.concat(reports).groupby(level=0).mean()
    
        print(f"{'='*25}")
        print(f"Model: {model_name}")
        print(f"{'-'*25}")
        print(f"Mean: {avg_report}")
        print()

Model: Logistic Regression
-------------------------
Mean:               precision    recall  f1-score      support
False          0.160646  0.089661  0.115088  2331.000000
True           0.572091  0.722067  0.638389  3929.000000
accuracy       0.486581  0.486581  0.486581     0.486581
macro avg      0.366368  0.405864  0.376738  6260.000000
weighted avg   0.418884  0.486581  0.443530  6260.000000

Model: Decision Tree
-------------------------
Mean:               precision    recall  f1-score      support
False          0.160646  0.089661  0.115088  2331.000000
True           0.572091  0.722067  0.638389  3929.000000
accuracy       0.486581  0.486581  0.486581     0.486581
macro avg      0.366368  0.405864  0.376738  6260.000000
weighted avg   0.418884  0.486581  0.443530  6260.000000

Model: KNN
-------------------------
Mean:               precision    recall  f1-score      support
False          0.372364  1.000000  0.542661  2331.000000
True           0.000000  0.000000  0.000000  

In [27]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    n_runs = 1000
    for model_name, model in models.items():
        reports = []
        all_y_true, all_y_pred = [], []
        
        for run in range(n_runs):
            dfs = generate_dataframe_for_modeling(downsample=False)
            report = logistic_regression_loo_classifier(dfs, by_subjects=False, as_dict=True, model=model)
            reports.append(pd.DataFrame(report).T)
    
        avg_report = pd.concat(reports).groupby(level=0).mean()
    
        print(f"{'='*25}")
        print(f"Model: {model_name}")
        print(f"{'-'*25}")
        print(f"Mean: {avg_report}")
        print()

Model: Logistic Regression
-------------------------
Mean:                               precision    recall  f1-score       support
BRCA1 pre-neoplastic           0.000000  0.000000  0.000000    476.000000
ER+ tumour                     0.412354  0.889387  0.563464   4909.000000
HER2+ tumour                   0.000000  0.000000  0.000000   2088.000000
Normal                         0.000000  0.000000  0.000000   2331.000000
Triple negative BRCA1 tumour   0.000000  0.000000  0.000000   1054.000000
Triple negative tumour         0.000000  0.000000  0.000000    273.000000
accuracy                       0.392238  0.392238  0.392238      0.392238
macro avg                      0.068726  0.148231  0.093911  11131.000000
weighted avg                   0.181856  0.392238  0.248499  11131.000000



KeyboardInterrupt: 